# GR-TFiLM diffssl LSTM — 02b mirror + GR curve as temporal-FiLM conditioning

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`06_output` modules and mounts Drive for the dataset. **Push local changes before running.**

## Idea — the 02b diffssl LSTM, with the GR curve added via a true TFiLM

This is a faithful mirror of `02b_sota_training/train_lstm_diffssl_tvc.ipynb`
(the diffssl `LSTM32TVC` recipe) with **one addition**: the exported gain-reduction
curve modulates the LSTM hidden features through a **temporal FiLM** (γ/β), the
mechanism from the nablafx fork (`modules.py::TFiLM`) — *not* the concat-style
`tvcond`.

```
raw dry ─┐                         4 static knobs
         │  tvcond (TVFiLMCond, UNCHANGED from 02b): pool(|x|)⊕knobs → cond_seq[16]
         └── cat(x, cond_seq) → main LSTM(17→32) → hidden
                                        │
              GR curve ─► GR-TFiLM: pool(GR) → blockLSTM → γ,β → modulate hidden
                                        │
                                   Linear(32→1) → tanh → wet
```

- **Static knobs** (threshold/attack/release/ratio): conditioned exactly like the
  SOTA — `TVFiLMCond` pools `|x|`, concatenates the 4 knobs, and the block-rate LSTM
  emits a `cond_seq` that is concatenated to the main LSTM input (`LSTM(17→32)`).
- **GR curve** (`GRTFiLMDiffSSLLSTM.gr_tfilm`): a temporal FiLM whose block-rate LSTM
  reads the (reduction-positive, peak-pooled) GR and produces per-block γ/β that
  modulate the 32-dim hidden features. This is the privileged, ground-truth envelope
  that diffssl had to *learn* from `|x|`, injected here as a proper FiLM modulation.

## Identical to 02b
- **Model core**: diffssl `LSTM32TVC` (`cond_type="tvcond"`, raw input, direct output + `tanh`).
- **Dataset / split**: Diff-SSL-G-Comp, 10 settings × 10 songs, seed 42
  (val = 1 song × all settings, test = held-out songs × lowest-threshold).
- **Training**: 3 s crops, `batch_size=16`, state reset every batch, TBPTT sub-steps
  of 4410; `0.5·L1 + 0.5·MR-STFT`, AdamW, fixed **100-epoch** budget.

**Speed tweaks** (deviate from 02b for a faster fixed-budget run): **cosine LR** over
the 100 epochs (`SCHEDULER="cosine"`), **bf16 autocast** around the LSTM forward
(`USE_AMP`), and no redundant full-crop MR-STFT recompute on train. Raise
`STEP_NUM_SAMPLES` or `CHECK_VAL_EVERY_N_EPOCH` to trade fidelity for more speed.

The other intentional deviation from diffssl is the split (shared with the other
`06_output` notebooks). The GR-TFiLM adds parameters (the temporal-FiLM LSTM emits
`2·hidden_size` for γ/β), so this model is larger than the ~8k-param SOTA baseline.

In [8]:
# -- 0. Dependencies ---------------------------------------------------
# This variant uses nablafx (TVFiLMMod). Pin numpy first so lightning/nablafx
# installs can't downgrade Colab's numpy 2.x and break torch. Install
# lightning/nablafx --no-deps so they can't clobber Colab's CUDA torch.
# `rational` / `frechet_audio_distance` are nablafx import-chain deps we never
# use here; stub both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")


numpy 2.0.2, torch 2.11.0+cu128


In [9]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module directory for this notebook (dataset_tfilm/model_tfilm/system_tfilm live here).
COND_DIR = os.path.join(REPO_ROOT, "06_output")
assert os.path.isfile(os.path.join(COND_DIR, "dataset_tfilm.py")), (
    f"Clone failed or stale: {COND_DIR}. Did you push local changes?"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "diffssl_gr_tfilm_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_tfilm", "model_tfilm", "system_tfilm", "splits", "amplitude_match"):
        del sys.modules[_name]

# repo root (for `src` + `nablafx`) + module dir (for dataset_tfilm/model_tfilm/...)
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), COND_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 619.58 KiB | 9.99 MiB/s, done.
From https://github.com/5aola/Virtual-Analogue-Compressor-Modelling
   b026dfc..d695007  main       -> origin/main
HEAD is now at d695007 update
REPO_ROOT  : /content/Virtual-Analogue-Compressor-Modelling
COND_DIR   : /content/Virtual-Analogue-Compressor-Modelling/06_output
DATA_ROOT  : /content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp
OUTPUT_DIR : /content/drive/Othercomputers/MacBook Air/data/diffssl_gr_tfilm_runs


In [10]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# The GR-TFiLM model needs dry (input) + GR curves (conditioning) + wet (target).
# Mirror the 02b cache: dry WAV per song, plus gr_curve (.pt) + wet WAV per pair.

import shutil
from dataset_tfilm import discover_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_gr_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# dry WAVs (one per song, shared across settings)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

# GR curves (.pt) + wet WAVs (-exported.wav), per (song, setting) pair
for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / "gr_curves" / p["setting"] / Path(p["gr"]).name)
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / p["setting"] / Path(p["wet"]).name)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")


Caching 10 songs x 10 settings (100 pairs) -> /content/Diff-SSL-G-Comp
Using local cache: /content/Diff-SSL-G-Comp


In [ ]:
# -- 3. Imports & hyper-parameters (02b LSTM32TVC + GR-TFiLM) ---------

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_tfilm as _dataset_tfilm
importlib.reload(_dataset_tfilm)
from dataset_tfilm import (
    BATCH_SIZE, SAMPLE_LENGTH, SAMPLE_RATE, GRCropDataModule, discover_gr_pairs,
)

import model_tfilm as _model_tfilm
importlib.reload(_model_tfilm)
from model_tfilm import GRTFiLMDiffSSLLSTM

import system_tfilm as _system_tfilm
importlib.reload(_system_tfilm)
from system_tfilm import GRTFiLMSystem

from splits import DIFFSSL_PARAM_RANGES, build_split_manifest
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split (identical to 02b / the other 06_output notebooks) --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1
N_TEST_SONGS = 2

# -- training (diffssl LSTM32TVC recipe: TBPTT sub-steps, state reset every
#    batch). Fixed 100-epoch budget with a cosine LR schedule + bf16 autocast
#    for speed (see GRTFiLMSystem). --
LR               = 1e-3
MAX_EPOCHS       = 100      # fixed budget == cosine T_max
STEP_NUM_SAMPLES = 4410     # diffssl TBPTT sub-step (0.1 s). Raise (e.g. 22050)
                            #   for fewer optimizer steps / faster epochs.
SCHEDULER        = "cosine"  # "cosine" (per-epoch, fits the fixed budget) | "plateau" | "none"
ETA_MIN          = 1e-6      # cosine floor
USE_AMP          = True      # bf16 autocast around the LSTM forward (cuda only)
CHECK_VAL_EVERY_N_EPOCH = 1  # raise to 2-5 to spend less time in validation

# -- model core (diffssl LSTM32TVC: tvcond on |x| + 4 static knobs) --
HIDDEN_SIZE     = 32        # LSTM32TVC
NUM_LAYERS      = 1
NUM_CONTROLS    = 4
TVCOND_DIM      = 16        # diffssl cond_dim (fixed at 16)
COND_BLOCK_SIZE = 128       # diffssl tvcond block (128/44100 ~= 2.9 ms)
COND_NUM_LAYERS = 1

# -- GR conditioning (temporal FiLM: pool(GR) -> blockLSTM -> gamma,beta) --
GR_TFILM_BLOCK_SIZE = 128
GR_TFILM_NUM_LAYERS = 1

# -- loss (SOTA waveform recipe, handled inside GRTFiLMSystem) --
#    0.5*L1 + 0.5*MR-STFT  (metrics: esr / rmse / mae / mse)

RUN_TAG    = "diffssl_lstm32_tvc_gr_tfilm"
RESUME_RUN = None

In [12]:
# -- 4. Preview split (must match 02b / 05) ---------------------------

preview = build_split_manifest(
    discover_gr_pairs(DATA_ROOT),
    seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")


Settings (10): ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2', 'threshold_-4_attack_10_release_0.1_ratio_2', 'threshold_-4_attack_1_release_0.4_ratio_10', 'threshold_-8_attack_30_release_0.8_ratio_4', 'threshold_0_attack_3_release_0.8_ratio_4', 'threshold_12_attack_3_release_0.8_ratio_2', 'threshold_4_attack_10_release_0.1_ratio_10', 'threshold_8_attack_1_release_0.1_ratio_10', 'threshold_8_attack_30_release_0.4_ratio_2']
Test settings (lowest T): ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2']
Train songs: ['BackroomInTulsa', 'Borderline', 'Electrvm', 'LivingLie', 'NosPalpitants', 'OpenFire', 'SongForJohn']
Val songs  : ['Ecstasy']
Test songs : ['Air', 'AncoraQui']
Pairs - train=70 val=10 test=4


In [13]:
# -- 5. Model size ----------------------------------------------------

model = GRTFiLMDiffSSLLSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
    gr_tfilm_block_size=GR_TFILM_BLOCK_SIZE, gr_tfilm_num_layers=GR_TFILM_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"GRTFiLMDiffSSLLSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\ncond_nn = TVFiLMCond (SOTA tvcond on |x| + {NUM_CONTROLS} knobs) | "
      f"gr_tfilm = temporal-FiLM on GR")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE} "
      f"({COND_BLOCK_SIZE/SAMPLE_RATE*1e3:.1f} ms) | GR-TFiLM block {GR_TFILM_BLOCK_SIZE}")


GRTFiLMDiffSSLLSTM: 25,185 params  (hidden=32, tvcond_dim=16, controls=4)
  cond_nn    1,472
  lstm       6,528
  gr_tfilm   17,152
  lin        33

cond_nn = TVFiLMCond (SOTA tvcond on |x| + 4 knobs) | gr_tfilm = temporal-FiLM on GR
Crop 132300 (3.00s) | 44100 Hz | TBPTT step 22050 | tvcond block 128 (2.9 ms) | GR-TFiLM block 128


In [ ]:
# -- 6. Train ---------------------------------------------------------

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

# Multi-worker loading: the model is small, so without this the GPU starves on
# the per-item soundfile seeks (dry + wet). Cap at 8 (matches 02b).
NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"gr_tfilm_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = GRCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS, n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "diffssl_lstm32_tvc + gr_temporal_film",
        "model_type": "GRTFiLMDiffSSLLSTM",
        "model_ref": "02b LSTM32TVC (tvcond) + nablafx-fork TFiLM on the GR curve",
        "dataset": "Diff-SSL-G-Comp", "setting": "multi (10 settings, tvcond on 4 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); GR via temporal FiLM (gamma/beta)",
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER, "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED, "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "gr_tfilm_block_size": GR_TFILM_BLOCK_SIZE,
                   "gr_tfilm_num_layers": GR_TFILM_NUM_LAYERS, "num_params": n_params},
        "loss": "0.5*L1 + 0.5*MR-STFT", "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GRTFiLMSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")


DataLoader num_workers: 8
NEW run: gr_tfilm_20260701_165533_diffssl_lstm32_tvc_gr_tfilm
Split seed     : 42
Train songs    : ['BackroomInTulsa', 'Borderline', 'Electrvm', 'LivingLie', 'NosPalpitants', 'OpenFire', 'SongForJohn']
Val songs      : ['Ecstasy']
Test songs     : ['Air', 'AncoraQui']
Test settings  : ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2'] (lowest threshold)
Pair counts    : train=70 val=10 test=4 / 100 total
GRCropDataset: 6500 crops from 70 pairs  [sample_length=132300, 325.0 min audio]


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


GRCropDataset: 850 crops from 10 pairs  [sample_length=132300, 42.5 min audio]
GRCropDataset: 284 crops from 4 pairs  [sample_length=132300, 14.2 min audio]
Train/val/test crops: 6500 / 850 / 284
Batches/epoch (train): 101  (batch_size=64)


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model  │ GRTFiLMDiffSSLLSTM      │ 25.2 K │ train │     0 │
│ 1 │ l1     │ L1Loss                  │      0 │ train │     0 │
│ 2 │ mrstft │ MultiResolutionSTFTLoss │      0 │ train │     0 │
└───┴────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 25.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 25.2 K                                                                                               
Total estimated model params size (MB): 0.101                                                                      
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
# -- 7. Test (held-out songs x lowest-threshold settings) -------------

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)


In [ ]:
# -- 8. Plot: dry input vs prediction vs target -----------------------
# Crops reset state every batch (diffssl regime), so just reset_states() before
# each batch and run the model. Plots a few examples from one val batch.

import matplotlib.pyplot as plt
import numpy as np
from system_tfilm import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred = system.model(dry.cuda(), gr.cuda(), params.cuda()).cpu()

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
n_plots = min(4, dry_np.shape[0])
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True, squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for ax, r in zip(axes[:, 0], range(n_plots)):
    ax.plot(t, dry_np[r, 0], label="Dry (input)", alpha=0.4, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    pred_mae = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {pred_mae:.4f} | ESR {float(esr_metric(tv, pv)):.4f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"GR-TFiLM diffssl LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.005)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"
